## 1. Import library
Pada bagian ini kita mengimpor library yang diperlukan:
- `re` untuk regex
- `pandas` untuk pengolahan data
- `matplotlib` untuk visualisasi

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

## 2. Membaca file data mahasiswa
Kita membaca file teks lalu menyimpan setiap baris non-kosong ke dalam list.

In [ ]:
with open("../data/hasil_generate/data_mahasiswaaa.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Jumlah baris data:", len(lines))
for i, line in enumerate(lines, start=1):
    print(f"{i}. {line}")

## 3. Fungsi validasi dasar
Bagian ini berisi beberapa fungsi untuk mengenali pola data:
- email
- domain email kampus tertentu
- tanggal lahir
- nomor telepon
- NIM/NRP
- username
- nama dan program studi

In [ ]:
ALLOWED_EMAIL_DOMAINS = ["pens.ac.id"]

def is_email(value):
    return re.fullmatch(r"[A-Za-z0-9._%+-]+@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", value) is not None

def is_allowed_campus_email(value):
    if not is_email(value):
        return False
    domain = value.split("@")[-1].lower()
    return domain in ALLOWED_EMAIL_DOMAINS

def is_date(value):
    return re.fullmatch(r"(0[1-9]|[12][0-9]|3[01])[-/](0[1-9]|1[0-2])[-/](19\d{2}|20\d{2})", value) is not None

def is_phone(value):
    # Nomor HP Indonesia: 08 + 8 sampai 11 digit berikutnya
    return re.fullmatch(r"08\d{8,11}", value) is not None

def is_nim_nrp(value):
    # Aturan ketat: hanya angka, panjang 9 sampai 10 digit
    return re.fullmatch(r"\d{9,10}", value) is not None

def is_username(value):
    return re.fullmatch(r"@[A-Za-z0-9_\.]+", value) is not None

def looks_like_name(value):
    return re.fullmatch(r"[A-Za-z ]{2,}", value) is not None

def looks_like_prodi(value):
    return re.fullmatch(r"[A-Za-z ]{3,}", value) is not None

## 4. Parsing setiap baris data
Setiap baris dipisahkan dengan koma, kemudian setiap token diklasifikasikan.

In [ ]:
parsed_data = []

for line in lines:
    parts = [p.strip() for p in line.split(",")]

    row = {
        "nama": None,
        "prodi": None,
        "telepon": None,
        "tanggal_lahir": None,
        "nim_nrp": None,
        "email": None,
        "username": None,
        "baris_asli": line
    }

    unknown_parts = []

    for part in parts:
        if looks_like_name(part) and row["nama"] is None:
            row["nama"] = part
        elif is_phone(part) and row["telepon"] is None:
            row["telepon"] = part
        elif is_date(part) and row["tanggal_lahir"] is None:
            row["tanggal_lahir"] = part
        elif is_nim_nrp(part) and row["nim_nrp"] is None:
            row["nim_nrp"] = part
        elif is_email(part) and row["email"] is None:
            row["email"] = part
        elif is_username(part) and row["username"] is None:
            row["username"] = part
        elif looks_like_prodi(part) and row["prodi"] is None:
            row["prodi"] = part
        else:
            unknown_parts.append(part)

    row["unknown_parts"] = unknown_parts
    parsed_data.append(row)

df = pd.DataFrame(parsed_data)
df

## 5. Validasi data dan penentuan status
Kita menambahkan:
- `status_data`: `valid` atau `perlu_cek`
- `catatan_validasi`: alasan kenapa data perlu dicek

In [ ]:
def validate_row(row):
    catatan = []

    if not row["nama"]:
        catatan.append("nama_tidak_terdeteksi")

    if row["email"]:
        if not is_allowed_campus_email(row["email"]):
            catatan.append("email_bukan_domain_kampus")
    else:
        catatan.append("email_tidak_ada")

    if row["telepon"]:
        if not is_phone(row["telepon"]):
            catatan.append("telepon_tidak_valid")
    else:
        catatan.append("telepon_tidak_ada")

    if row["nim_nrp"]:
        if not is_nim_nrp(row["nim_nrp"]):
            catatan.append("nim_nrp_tidak_valid")
    else:
        catatan.append("nim_nrp_tidak_ada")

    if row["tanggal_lahir"]:
        if not is_date(row["tanggal_lahir"]):
            catatan.append("tanggal_tidak_valid")
    else:
        catatan.append("tanggal_tidak_ada")

    status = "valid" if len(catatan) == 0 else "perlu_cek"
    return pd.Series([status, "; ".join(catatan)])

df[["status_data", "catatan_validasi"]] = df.apply(validate_row, axis=1)
df

## 6. Pisahkan data valid dan data tidak valid

In [ ]:
df_valid = df[df["status_data"] == "valid"].copy()
df_tidak_valid = df[df["status_data"] == "perlu_cek"].copy()

print("Jumlah data valid     :", len(df_valid))
print("Jumlah data perlu cek :", len(df_tidak_valid))

## 7. Simpan ke dua file CSV terpisah
Hasil dibagi menjadi:
- `data_mahasiswa_valid.csv`
- `data_mahasiswa_tidak_valid.csv`

In [ ]:
df_valid.to_csv("data_mahasiswa_valid.csv", index=False, encoding="utf-8")
df_tidak_valid.to_csv("data_mahasiswa_tidak_valid.csv", index=False, encoding="utf-8")

print("Berhasil menyimpan file:")
print("- data_mahasiswa_valid.csv")
print("- data_mahasiswa_tidak_valid.csv")

## 8. Tampilkan data hasil validasi

In [ ]:
print("=== DATA VALID ===")
display(df_valid)

print("=== DATA PERLU CEK ===")
display(df_tidak_valid)

## 9. Visualisasi sederhana: jumlah status data

In [ ]:
status_counts = df["status_data"].value_counts()

plt.figure(figsize=(6, 4))
status_counts.plot(kind="bar")
plt.title("Jumlah Data Berdasarkan Status")
plt.xlabel("Status Data")
plt.ylabel("Jumlah")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Visualisasi sederhana: kelengkapan kolom

In [ ]:
kolom_cek = ["nama", "prodi", "telepon", "tanggal_lahir", "nim_nrp", "email", "username"]
kelengkapan = df[kolom_cek].notna().sum()

plt.figure(figsize=(8, 4))
kelengkapan.plot(kind="bar")
plt.title("Jumlah Data yang Terisi per Kolom")
plt.xlabel("Kolom")
plt.ylabel("Jumlah Terisi")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Ringkasan hasil

In [ ]:
print("Ringkasan:")
print(f"Total baris data : {len(df)}")
print(f"Data valid       : {len(df_valid)}")
print(f"Data perlu cek   : {len(df_tidak_valid)}")

print("\nDistribusi catatan validasi:")
print(df["catatan_validasi"].value_counts())

## 12. Studi kasus tambahan untuk latihan mandiri
Beberapa ide pengembangan lanjutan:
1. Tambahkan validasi beberapa domain kampus sekaligus.
2. Tambahkan normalisasi huruf besar-kecil untuk program studi.
3. Buat fungsi untuk memperbaiki format tanggal menjadi standar yang sama.
4. Tambahkan ekspor ke Excel.
5. Buat dashboard sederhana dari data validasi.